<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_1_imagenes_estructura.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_1_imagenes_estructura.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F03_vision%2Fnotebooks%2Fsesion_03_1_imagenes_estructura.ipynb)

# SI7011 · Sesión 03 · 1. Imágenes y estructura
**Juan David Martínez Vargas · Universidad EAFIT**

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | una imagen como tensor $C \times H \times W$ | formas, canales y normalización por canal |
| 2 | MLP sobre la imagen aplanada (línea base) | parámetros y accuracy de validación |
| 3 | una CNN pequeña | menos parámetros, más accuracy |
| 4 | permutar los píxeles | qué arquitectura lo nota y por qué |

Corre en CPU en 5–8 minutos (Colab, Kaggle, Lightning o local).

In [ ]:
import json, time
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
from torch import nn
from torchvision import datasets

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
int_epochs = lambda: plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
print('PyTorch', torch.__version__, '· device', device)

## Datos: CIFAR-10
60 000 imágenes en color de 32×32, en 10 clases. Usamos **10 000** para entrenar y **5 000** para validar, tomadas del conjunto de entrenamiento, y el conjunto de prueba oficial se deja para el final. Son los mismos datos en los notebooks 1, 2, 3 y 5 de la sesión.

In [ ]:
train_set = datasets.CIFAR10('data', train=True, download=True)
test_set = datasets.CIFAR10('data', train=False, download=True)
CLASSES = train_set.classes

to_tensor = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float() / 255   # N×H×W×C uint8 → N×C×H×W en [0, 1]
X = to_tensor(train_set.data)
y = torch.tensor(train_set.targets)
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
X_train, y_train = X[perm[:10_000]], y[perm[:10_000]]
X_val, y_val = X[perm[45_000:]], y[perm[45_000:]]
X_test, y_test = to_tensor(test_set.data), torch.tensor(test_set.targets)
X_train.shape, X_val.shape, X_test.shape

## 1. Una imagen es un tensor $C \times H \times W$
`torchvision` entrega cada imagen como un arreglo $H \times W \times C$ de enteros entre 0 y 255. PyTorch espera **canales primero**: $C \times H \times W$, y un lote agrega la dimensión $N$ adelante.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(10, 1.7))
for ax, img, lab in zip(axes, X_train[:8], y_train[:8]):
    ax.imshow(img.permute(1, 2, 0)); ax.set_title(CLASSES[lab], fontsize=8); ax.axis('off')
plt.show()

In [ ]:
x = X_train[0]                                   # una imagen: 3 × 32 × 32
fig, axes = plt.subplots(1, 4, figsize=(8, 2.2))
axes[0].imshow(x.permute(1, 2, 0)); axes[0].set_title('imagen')
for ax, c, name, cmap in zip(axes[1:], range(3), 'RGB', ['Reds', 'Greens', 'Blues']):
    ax.imshow(x[c], cmap=cmap, vmin=0, vmax=1); ax.set_title(f'canal {name}')
for ax in axes: ax.axis('off')
plt.show()
print('una posición, los tres canales: x[:, 10, 20] =', x[:, 10, 20])

Cada canal es una matriz de 32×32: el color de un píxel son **tres números**, uno por canal, en la misma posición $(i, j)$.

Normalizamos **por canal**, con la media y la desviación estándar de cada canal calculadas **solo con entrenamiento**. Es la misma regla de la sesión 02, ahora con un vector de 3 valores en lugar de un número.

In [ ]:
mean = X_train.mean(dim=(0, 2, 3))             # promedia sobre N, H, W → un valor por canal
std = X_train.std(dim=(0, 2, 3))
print('media por canal', mean, '· std por canal', std)
norm = lambda X: (X - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)
X_train, X_val, X_test = norm(X_train), norm(X_val), norm(X_test)
X_train.mean(dim=(0, 2, 3)), X_train.std(dim=(0, 2, 3))

## Herramientas: entrenar, evaluar y graficar
Es el loop de la sesión 02. Hay dos cambios: el lote se mueve a `device` y la evaluación va por trozos de 1 000 imágenes, para no llenar la memoria. `fit` recibe los datos como argumento porque en la sección 4 entrenaremos con imágenes permutadas.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_epoch(model, opt, X, y, batch_size=128):
    model.train()
    perm = torch.randperm(len(X))
    total = 0.0
    for i in range(0, len(X), batch_size):
        b = perm[i:i + batch_size]
        xb, yb = X[b].to(device), y[b].to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(b)
    return total / len(X)

@torch.no_grad()
def evaluate(model, X, y, chunk=1000):
    model.eval()
    logits = torch.cat([model(X[i:i + chunk].to(device)).cpu() for i in range(0, len(X), chunk)])
    return loss_fn(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

In [ ]:
def fit(model, epochs=10, lr=1e-3, data=None):
    X_tr, y_tr, X_va, y_va = data or (X_train, y_train, X_val, y_val)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {'loss': [], 'val_loss': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(epochs):
        hist['loss'].append(train_epoch(model, opt, X_tr, y_tr))
        val_loss, val_acc = evaluate(model, X_va, y_va)
        hist['val_loss'].append(val_loss); hist['val_acc'].append(val_acc)
    print(f"época {epochs}: loss {hist['loss'][-1]:.3f} · val_loss {val_loss:.3f} · val_acc {val_acc:.3f} · {time.time() - t0:.0f} s")
    return hist

def plot_curves(hists, key='val_acc', title=''):
    for name, h in hists.items():
        plt.plot(range(1, len(h[key]) + 1), h[key], label=name)
    plt.xlabel('época'); plt.ylabel(key); plt.title(title); plt.legend(); int_epochs(); plt.show()

n_params = lambda m: sum(p.numel() for p in m.parameters())

## 2. Línea base: un MLP sobre la imagen aplanada
`nn.Flatten()` convierte cada imagen $3 \times 32 \times 32$ en un vector de 3 072 números. Desde ese momento el modelo ya no sabe qué píxeles son vecinos.

In [ ]:
def mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(3 * 32 * 32, 512), nn.ReLU(),
        nn.Linear(512, 256), nn.ReLU(),
        nn.Linear(256, 10),
    )

torch.manual_seed(0)
model_mlp = mlp()
print(f'MLP: {n_params(model_mlp):,} parámetros')
h_mlp = fit(model_mlp)

Casi todos los parámetros están en la primera capa: $3072 \times 512 + 512 \approx 1.6$ M (diapositiva *Flattening an image into an MLP*).

## 3. Una CNN pequeña
**Cambia la arquitectura:** tres bloques `Conv2d` → `ReLU` → `MaxPool2d`. Cada convolución mira vecindarios de 3×3 y usa los mismos pesos en todas las posiciones. Cada `MaxPool2d(2)` reduce a la mitad la resolución.

In [ ]:
def small_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 32 × 16 × 16
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 64 × 8 × 8
        nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 64 × 4 × 4
        nn.Flatten(),
        nn.Linear(64 * 4 * 4, 10),
    )

torch.manual_seed(0)
model_cnn = small_cnn()
xb = X_train[:1]
for layer in model_cnn:                          # sigue la forma del tensor capa por capa
    xb = layer(xb)
    if not isinstance(layer, nn.ReLU):
        print(f'{layer.__class__.__name__:10s} → {tuple(xb.shape[1:])}')
print(f'CNN: {n_params(model_cnn):,} parámetros')

In [ ]:
h_cnn = fit(model_cnn)
plot_curves({'MLP': h_mlp, 'CNN pequeña': h_cnn}, title='Accuracy de validación')

La CNN tiene unas **25 veces menos parámetros** y aun así le gana al MLP. La diferencia no está en la capacidad sino en el **supuesto**: los pesos de cada convolución se reutilizan en toda la imagen, y cada salida depende solo de un vecindario.

## 4. Permutar los píxeles
Elegimos **una** permutación fija $\pi$ de las 3 072 posiciones y la aplicamos a **todas** las imágenes, en entrenamiento, validación y prueba. Para un humano, las imágenes dejan de ser reconocibles.

**Antes de correr:** ¿qué le pasa a la accuracy del MLP? ¿Y a la de la CNN? (diapositiva *Prediction before experiment*).

In [ ]:
pi = torch.randperm(3 * 32 * 32, generator=torch.Generator().manual_seed(1))
permute = lambda X: X.reshape(len(X), -1)[:, pi].reshape(X.shape)
P_train, P_val, P_test = permute(X_train), permute(X_val), permute(X_test)

show = lambda t: ((t * std.view(3, 1, 1) + mean.view(3, 1, 1)).clamp(0, 1)).permute(1, 2, 0)
fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for k in range(6):
    axes[0, k].imshow(show(X_train[k])); axes[1, k].imshow(show(P_train[k]))
    axes[0, k].set_title(CLASSES[y_train[k]], fontsize=8)
for ax in axes.flat: ax.axis('off')
axes[0, 0].set_ylabel('original'); axes[1, 0].set_ylabel('permutada')
plt.show()

In [ ]:
torch.manual_seed(0)
h_mlp_p = fit(mlp(), data=(P_train, y_train, P_val, y_val))
torch.manual_seed(0)
h_cnn_p = fit(small_cnn(), data=(P_train, y_train, P_val, y_val))

In [ ]:
plot_curves({'MLP': h_mlp, 'MLP, píxeles permutados': h_mlp_p,
             'CNN': h_cnn, 'CNN, píxeles permutados': h_cnn_p}, title='Accuracy de validación')

El **MLP no se entera**: entrenar sobre $x_\pi$ equivale a entrenar sobre $x$ con las columnas de la primera matriz de pesos reordenadas, $W x_\pi = W_{\pi^{-1}}\, x$. Las curvas difieren solo por el azar de la inicialización y del orden de los lotes.

La **CNN cae** hasta quedar cerca del MLP, o por debajo. Sus kernels de 3×3 suponen que los píxeles vecinos están relacionados, y la permutación rompe exactamente ese supuesto. El supuesto ayuda cuando es cierto y no ayuda nada cuando es falso.

In [ ]:
resultados = {name: max(h['val_acc']) for name, h in
              {'mlp': h_mlp, 'mlp_permutado': h_mlp_p, 'cnn': h_cnn, 'cnn_permutado': h_cnn_p}.items()}
json.dump(resultados, open('resultados_permutacion.json', 'w'), indent=1)
{k: round(v, 3) for k, v in resultados.items()}

## Evaluación en prueba
Una sola vez, con la CNN sobre las imágenes originales.

In [ ]:
test_loss, test_acc = evaluate(model_cnn, X_test, y_test)
print(f'CNN · prueba: loss {test_loss:.3f} · acc {test_acc:.3f}')

## Tu turno · 10 minutos
Copia la celda de la sección que corresponda, cambia **una** cosa y superpón la curva nueva con `plot_curves`.

1. Permuta solo las **posiciones** (las mismas 1 024 para los tres canales), de modo que cada píxel conserve su color. ¿Le va mejor a la CNN que con la permutación completa? ¿Por qué?
2. Dale al MLP una capa oculta más de 512. ¿Alcanza a la CNN? ¿Cuántos parámetros tiene ahora?
3. Reduce los canales de la CNN a la mitad (16, 32, 32). ¿Cuánto bajan los parámetros y cuánto la accuracy?
4. Sin reentrenar, desplaza las imágenes de validación 3 píxeles a la derecha con `torch.roll(X_val, 3, dims=3)` y evalúa ambos modelos. ¿Cuál se degrada menos? (lo retomamos en el notebook 2, *equivarianza*).

**Para pensar.** En la sección 4 el MLP y la CNN vieron exactamente la misma información. ¿Por qué entonces a uno le importa la permutación y al otro no?

In [ ]:
# Experimento 1: permutar solo las posiciones espaciales

### Referencias
- LeCun et al. (1998), *Gradient-Based Learning Applied to Document Recognition*, Proc. IEEE.
- Krizhevsky (2009), *Learning Multiple Layers of Features from Tiny Images* (CIFAR-10).
- Chollet y Watson (2025), *Deep Learning with Python*, 3.ª ed., cap. 8 (el formato de comparar una sola variable a la vez viene de ahí).